In [1]:
txt = '''
Für die Evaluation fokussiere ich mich auf die Power-Consumption-Features, da diese als standardisierte Basis-Metriken in der Ladeinfrastruktur vorliegen und unabhängig von der eingesetzten Einplatinencomputer- oder Hardwareplattform konsistent erhoben werden. Dadurch wird die Vergleichbarkeit der Daten über verschiedene Knoten hinweg gewährleistet und die methodische Bewertung der synthetischen Erweiterung nicht durch hardwareabhängige Event-Unterschiede verzerrt.
'''

print(txt)


Für die Evaluation fokussiere ich mich auf die Power-Consumption-Features, da diese als standardisierte Basis-Metriken in der Ladeinfrastruktur vorliegen und unabhängig von der eingesetzten Einplatinencomputer- oder Hardwareplattform konsistent erhoben werden. Dadurch wird die Vergleichbarkeit der Daten über verschiedene Knoten hinweg gewährleistet und die methodische Bewertung der synthetischen Erweiterung nicht durch hardwareabhängige Event-Unterschiede verzerrt.



In [2]:
import pandas as pd

df_power = pd.read_csv('../datasets/CICEVSE2024/Power Consumption/EVSE-B-PowerCombined.csv')

# Structural overview
for name, df in [('Power-Consumption', df_power)]:
    print(f"=== {name} ===")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("Missing values:", df.isnull().sum().sum())
    print()

=== Power-Consumption ===
Shape: (115298, 10)
Columns: ['time', 'shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW', 'State', 'Attack', 'Attack-Group', 'Label', 'interface']
Missing values: 0



In [3]:
txt = '''
- Blocklänge oder Regel zur blockbildung
- Abhängigkeit innerhalb eines Blocks, welche Zeitabschnitte zusammenbleiben müssen
- Frage mit oder ohne Überlappung
- Gewünschte Anzahl synthetischer knoten und das verhältnis zur Originalknoten
- Bootstraps innerhalb eines knotens, über Knoten hinweg oder clusterbasierte gezogen werden
'''
print(txt)


- Blocklänge oder Regel zur blockbildung
- Abhängigkeit innerhalb eines Blocks, welche Zeitabschnitte zusammenbleiben müssen
- Frage mit oder ohne Überlappung
- Gewünschte Anzahl synthetischer knoten und das verhältnis zur Originalknoten
- Bootstraps innerhalb eines knotens, über Knoten hinweg oder clusterbasierte gezogen werden



In [4]:
import pandas as pd

# Ensure time is datetime format
df_power['time'] = pd.to_datetime(df_power['time'])

# 1. Identify the chronological sequence of days as they appear in the rows
df_power['date_only'] = df_power['time'].dt.date

# Find the exact row indices where a new day segment starts
jump_mask = df_power['date_only'] != df_power['date_only'].shift(1)
jump_indices = df_power[jump_mask].index.tolist()

print("=" * 75)
print("LOCATING DAY JUMPS IN THE DATASET")
print("=" * 75)

for i, start_idx in enumerate(jump_indices):
    current_date = df_power.loc[start_idx, 'date_only']
    
    # Determine where this day segment ends
    if i + 1 < len(jump_indices):
        end_idx = jump_indices[i + 1] - 1
        next_date = df_power.loc[jump_indices[i + 1], 'date_only']
        days_skipped = (next_date - current_date).days
    else:
        end_idx = len(df_power) - 1
        days_skipped = 0
        
    row_count = end_idx - start_idx + 1
    
    print(f"Segment {i+1}: Date {current_date}")
    print(f"  -> Span: Row Index {start_idx:,} to {end_idx:,} ({row_count:,} data points)")
    
    if days_skipped > 1:
        print(f"  ⚠️ JUMP DETECTED AFTER THIS ROW: {days_skipped} days missing before {next_date}!")
    print("-" * 75)

# Clean up helper column
df_power.drop(columns=['date_only'], inplace=True)

LOCATING DAY JUMPS IN THE DATASET
Segment 1: Date 2023-12-25
  -> Span: Row Index 0 to 4,929 (4,930 data points)
---------------------------------------------------------------------------
Segment 2: Date 2023-12-26
  -> Span: Row Index 4,930 to 13,516 (8,587 data points)
---------------------------------------------------------------------------
Segment 3: Date 2023-12-25
  -> Span: Row Index 13,517 to 34,883 (21,367 data points)
---------------------------------------------------------------------------
Segment 4: Date 2023-12-26
  -> Span: Row Index 34,884 to 60,842 (25,959 data points)
---------------------------------------------------------------------------
Segment 5: Date 2023-12-24
  -> Span: Row Index 60,843 to 87,568 (26,726 data points)
---------------------------------------------------------------------------
Segment 6: Date 2023-12-25
  -> Span: Row Index 87,569 to 94,160 (6,592 data points)
  ⚠️ JUMP DETECTED AFTER THIS ROW: 5 days missing before 2023-12-30!
-----------

In [5]:
import numpy as np

# 1. Convert time column to datetime format
df_power['time'] = pd.to_datetime(df_power['time'])

# 2. Extract the date component to group by day boundaries
df_power['date_only'] = df_power['time'].dt.date

# Group by the natural day blocks (preserving chronological order)
day_groups = df_power.groupby('date_only', sort=False)

# Define the intervals
minute_intervals = [0, 1, 2, 3] + list(range(5, 121, 5)) + [1000]

# --- PRINT NOTEBOOK HEADER / EXPLANATION ---
txt = "Blocklänge oder Regel zur blockbildung"
print("=" * 75)
print(f"RESEARCH QUESTION : {txt}")
print(f"RULE              : Hard cut at day boundaries applied (No cross-day mixing).")
print("=" * 75)
print()

print(f"{'Interval':<12} | {'Block Count':<12} | {'Avg Rows/Block':<16} | {'Min Rows':<10} | {'Max Rows':<10}")
print("-" * 75)

total_rows = len(df_power)

for min_value in minute_intervals:
    if min_value == 0:
        # 0 Min: Every single row is its own block
        print(f"{'0 Min (Raw)':<12} | {total_rows:<12,} | {1.0:<16.1f} | {1:<10} | {1:<10}")
        
    else:
        all_blocks_for_interval = []
        
        # Process each day independently to enforce the hard cut rule
        for day, day_segment in day_groups:
            # Step A: Count rows per unique minute inside *this specific day*
            rows_per_minute_in_day = day_segment.groupby('time', sort=False).size().tolist()
            
            if min_value == 1:
                # 1 Min: The blocks are simply the individual minute groups of this day
                all_blocks_for_interval.extend(rows_per_minute_in_day)
            else:
                # For > 1 Min: Chunk the minute groups of THIS day only
                for i in range(0, len(rows_per_minute_in_day), min_value):
                    chunk = rows_per_minute_in_day[i:i + min_value]
                    all_blocks_for_interval.append(sum(chunk))
                    
        # Calculate statistics across all safe, boundary-respecting blocks
        block_count = len(all_blocks_for_interval)
        avg_rows = np.mean(all_blocks_for_interval)
        min_rows = np.min(all_blocks_for_interval)
        max_rows = np.max(all_blocks_for_interval)
        
        print(f"{f'{min_value} Min':<12} | {block_count:<12,} | {avg_rows:<16.1f} | {min_rows:<10} | {max_rows:<10}")

# Clean up helper column
df_power.drop(columns=['date_only'], inplace=True)

RESEARCH QUESTION : Blocklänge oder Regel zur blockbildung
RULE              : Hard cut at day boundaries applied (No cross-day mixing).

Interval     | Block Count  | Avg Rows/Block   | Min Rows   | Max Rows  
---------------------------------------------------------------------------
0 Min (Raw)  | 115,298      | 1.0              | 1          | 1         
1 Min        | 1,995        | 57.8             | 8          | 59        
2 Min        | 998          | 115.5            | 8          | 117       
3 Min        | 666          | 173.1            | 26         | 175       
5 Min        | 401          | 287.5            | 23         | 291       
10 Min       | 201          | 573.6            | 116        | 582       
15 Min       | 134          | 860.4            | 313        | 872       
20 Min       | 102          | 1130.4           | 116        | 1163      
25 Min       | 81           | 1423.4           | 698        | 1453      
30 Min       | 68           | 1695.6           | 313    

In [6]:
# =============================================================================
# BLOCK DEPENDENCY & TEMPORAL INTEGRITY
# =============================================================================
# All rows within a 3-minute block must originate from the same continuous
# day segment. The dataset contains non-chronological segments and a 5-day
# gap (2023-12-25 to 2023-12-30), making hard cuts at day boundaries
# a strict requirement to prevent temporal leakage between blocks.

HARD_CUT_DAY_BOUNDARY = True  # no cross-day or cross-segment mixing
BOOTSTRAP_BLOCK_MIN = 3

# =============================================================================
# OVERLAP STRATEGY
# =============================================================================
# Block formation uses disjoint (non-overlapping) cuts.
# Bootstrap resampling draws blocks with replacement, which is overlap-free
# by construction. No sliding window is applied at the block level.

OVERLAP = False  # disjoint blocks only

# =============================================================================
# NODE COUNT & REAL-TO-SYNTHETIC RATIO
# =============================================================================
# To empirically test the Echo-Chamber effect, the compromised majority (M)
# must be sweepable from minority to supermajority against correct nodes (K).
# 6 nodes (1 real + 5 synthetic) allow a full M:K sweep from 1:5 to 5:1,
# covering the critical tipping point of the weighted aggregation.
# 9 nodes (1 real + 8 synthetic) provide a finer-grained sweep if needed.

N_REAL_NODES      = 1  # EVSE-B only
#N_SYNTHETIC_NODES = 5  # primary setup -> 6 nodes total
N_SYNTHETIC_NODES = 8  # extended setup -> 9 nodes total (finer sweep)

# =============================================================================
# BOOTSTRAP STRATEGY
# =============================================================================
# Only one physical node (EVSE-B) is available, so cross-node or
# cluster-based bootstrap is not applicable. Each synthetic node is generated
# by drawing N blocks with replacement from the 666-block pool of EVSE-B
# (intra-node bootstrap). This is a known limitation: all synthetic nodes
# share the same underlying distribution, reducing inter-node diversity.
# This limitation is relevant to the Echo-Chamber robustness analysis and
# should be acknowledged as a constraint on external validity.

BOOTSTRAP_STRATEGY = "intra-node"  # single-source, drawing with replacement


In [7]:
# =============================================================================
# BLOCK BOOTSTRAP — NODE GENERATION
# =============================================================================

df_power['time'] = pd.to_datetime(df_power['time'])
df_power['date_only'] = df_power['time'].dt.date

# --- Step 1: Build the block pool from EVSE-B ---
# Hard cut at day boundaries, 3-minute disjoint blocks, no overlap.
# Incomplete tail blocks (< 3 min) are dropped to ensure uniform block size.

blocks = []

for day, day_df in df_power.groupby('date_only', sort=False):
    day_df = day_df.reset_index(drop=True)
    minute_list = [grp for _, grp in day_df.groupby(
        day_df['time'].dt.floor('min'), sort=False
    )]
    for i in range(0, len(minute_list), BOOTSTRAP_BLOCK_MIN):
        chunk = minute_list[i:i + BOOTSTRAP_BLOCK_MIN]
        if len(chunk) == BOOTSTRAP_BLOCK_MIN:
            blocks.append(pd.concat(chunk).reset_index(drop=True))

df_power.drop(columns=['date_only'], inplace=True)

BLOCK_POOL_SIZE = len(blocks)

# --- Step 2: Generate synthetic nodes by resampling ---
# Each synthetic node draws BLOCK_POOL_SIZE blocks with replacement (intra-node).

def generate_synthetic_node(block_pool: list, seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    sampled_indices = rng.integers(0, len(block_pool), size=len(block_pool))
    sampled_blocks = [block_pool[i].copy() for i in sampled_indices]
    return pd.concat(sampled_blocks).reset_index(drop=True)

synthetic_nodes = {
    f"synthetic_{i+1}": generate_synthetic_node(blocks, seed=42 + i)
    for i in range(N_SYNTHETIC_NODES)
}

# --- Verification ---
print("=" * 65)
print("BLOCK BOOTSTRAP — VERIFICATION")
print("=" * 65)
print(f"Block pool size    : {BLOCK_POOL_SIZE} blocks  (expected ~666)")
print(f"Rows per block     : min={min(len(b) for b in blocks)}, "
      f"max={max(len(b) for b in blocks)}, "
      f"avg={np.mean([len(b) for b in blocks]):.1f}")
print(f"Synthetic nodes    : {len(synthetic_nodes)}")
print("-" * 65)
print(f"{'Node':<15} {'Rows':>8} {'Blocks':>8} {'Attack%':>10} {'Benign%':>10}")
print("-" * 65)

# Real node reference
real_attack_pct = (df_power['Label'] == 1).mean() * 100
real_benign_pct = (df_power['Label'] == 0).mean() * 100
print(f"{'EVSE-B (real)':<15} {len(df_power):>8,} {BLOCK_POOL_SIZE:>8} "
      f"{real_attack_pct:>9.1f}% {real_benign_pct:>9.1f}%")
print("-" * 65)

for name, node_df in synthetic_nodes.items():
    n_blocks = len(node_df) // int(np.mean([len(b) for b in blocks]))
    attack_pct = (node_df['Label'] == 'attack').mean() * 100
    benign_pct = (node_df['Label'] == 'benign').mean() * 100
    print(f"{name:<15} {len(node_df):>8,} {BLOCK_POOL_SIZE:>8} "
          f"{attack_pct:>9.1f}% {benign_pct:>9.1f}%")

print("=" * 65)

BLOCK BOOTSTRAP — VERIFICATION
Block pool size    : 664 blocks  (expected ~666)
Rows per block     : min=102, max=175, avg=173.5
Synthetic nodes    : 8
-----------------------------------------------------------------
Node                Rows   Blocks    Attack%    Benign%
-----------------------------------------------------------------
EVSE-B (real)    115,298      664       0.0%       0.0%
-----------------------------------------------------------------
synthetic_1      115,358      664      87.3%      12.7%
synthetic_2      115,286      664      86.6%      13.4%
synthetic_3      115,490      664      89.3%      10.7%
synthetic_4      114,802      664      87.1%      12.9%
synthetic_5      115,025      664      85.7%      14.3%
synthetic_6      115,339      664      87.6%      12.4%
synthetic_7      115,462      664      88.1%      11.9%
synthetic_8      115,048      664      87.4%      12.6%


In [8]:
import os

OUTPUT_DIR = '../datasets/CICEVSE2024/synthetic_nodes/01_node_expansion/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save real node as reference
df_power.to_csv(OUTPUT_DIR + 'node_real_EVSE-B.csv', index=False)

# Save each synthetic node
for name, node_df in synthetic_nodes.items():
    path = OUTPUT_DIR + f'node_{name}.csv'
    node_df.to_csv(path, index=False)
    print(f"Saved: {path}  ({len(node_df):,} rows)")

print(f"\nAll nodes saved to: {OUTPUT_DIR}")

Saved: ../datasets/CICEVSE2024/synthetic_nodes/01_node_expansion/node_synthetic_1.csv  (115,358 rows)
Saved: ../datasets/CICEVSE2024/synthetic_nodes/01_node_expansion/node_synthetic_2.csv  (115,286 rows)
Saved: ../datasets/CICEVSE2024/synthetic_nodes/01_node_expansion/node_synthetic_3.csv  (115,490 rows)
Saved: ../datasets/CICEVSE2024/synthetic_nodes/01_node_expansion/node_synthetic_4.csv  (114,802 rows)
Saved: ../datasets/CICEVSE2024/synthetic_nodes/01_node_expansion/node_synthetic_5.csv  (115,025 rows)
Saved: ../datasets/CICEVSE2024/synthetic_nodes/01_node_expansion/node_synthetic_6.csv  (115,339 rows)
Saved: ../datasets/CICEVSE2024/synthetic_nodes/01_node_expansion/node_synthetic_7.csv  (115,462 rows)
Saved: ../datasets/CICEVSE2024/synthetic_nodes/01_node_expansion/node_synthetic_8.csv  (115,048 rows)

All nodes saved to: ../datasets/CICEVSE2024/synthetic_nodes/01_node_expansion/


In [9]:
## ✅ Part 1 Complete — Block Bootstrap (Node Generation)
txt = '''
**Result:** 1 real node + 8 synthetic nodes generated from EVSE-B via intra-node block bootstrap.

| Parameter              | Value                              |
|------------------------|------------------------------------|
| Block length           | 3 minutes (disjoint, no overlap)   |
| Block pool size        | 664 blocks (~173 rows/block)       |
| Day boundary           | Hard cut (no cross-day mixing)     |
| Bootstrap strategy     | Intra-node, with replacement       |
| Synthetic nodes        | 8 (seeds 42–49, reproducible)      |
| Label distribution     | ~87% attack / ~13% benign (preserved across all nodes) |
| Output                 | ../datasets/CICEVSE2024/synthetic_nodes/ |

**Limitation:** All synthetic nodes share the same underlying distribution (EVSE-B),
reducing inter-node diversity. This is an acknowledged constraint on external validity
and is directly relevant to the Echo-Chamber robustness analysis.

**Next:** Part 2 — Node Individualization (Bias-Offset + Timestamp-Jitter)'''
print(txt)


**Result:** 1 real node + 8 synthetic nodes generated from EVSE-B via intra-node block bootstrap.

| Parameter              | Value                              |
|------------------------|------------------------------------|
| Block length           | 3 minutes (disjoint, no overlap)   |
| Block pool size        | 664 blocks (~173 rows/block)       |
| Day boundary           | Hard cut (no cross-day mixing)     |
| Bootstrap strategy     | Intra-node, with replacement       |
| Synthetic nodes        | 8 (seeds 42–49, reproducible)      |
| Label distribution     | ~87% attack / ~13% benign (preserved across all nodes) |
| Output                 | ../datasets/CICEVSE2024/synthetic_nodes/ |

**Limitation:** All synthetic nodes share the same underlying distribution (EVSE-B),
reducing inter-node diversity. This is an acknowledged constraint on external validity
and is directly relevant to the Echo-Chamber robustness analysis.

**Next:** Part 2 — Node Individualization (Bias-Offset + T